# 02_1. Feature rationale and expected signs / Обоснование признаков и ожидаемые знаки

**EN.** Step 1 of feature selection (methodology, section 5). The Y1 population is loans that are **current at the observation date t** (DPD = 0) **and have never been in default before t**, and there are **59 candidate features**. Only the first default of a loan counts: a loan that was already 90+ days past due before t and has since cured is excluded from the month of its first default on. Such repeat defaults are easy to predict (the loan already has a default history); the early-warning question is about loans that look healthy. Delinquency-history features (`dpd`, `dpd_max_6m`, `dpd_max_12m`, `dpd_mean_12m`, `n_30p_6m`, `n_30p_12m`, `n_60p_12m`, `n_cure_12m`, `months_since_30p`, `ever_30p`) are not candidates: a missed payment is already a visible step on the way to default, so these features are too close to the event (almost leakage). The model must warn about *hidden deterioration*, before any delinquency is visible. For each of the 59 candidates we write down *before looking at any relationship with the target*: its group, its economic meaning, and the **expected sign** of its link with risk. Later notebooks compare the observed sign with this expectation. If they disagree, we investigate (leakage, interaction, data error) — we never rewrite the expectation.

**RU.** Шаг 1 отбора признаков (методология, раздел 5). Популяция Y1 — кредиты **без просрочки на дату наблюдения t** (DPD = 0), **ни разу не бывшие в дефолте до t**; кандидатов **59**. Учитывается только первый дефолт кредита: кредит, который уже был в просрочке 90+ дней до t и потом вылечился, исключается начиная с месяца первого дефолта. Повторный дефолт предсказать легко (у кредита уже есть история дефолта), а задача раннего предупреждения — о кредитах, которые выглядят здоровыми. Признаки истории просрочек (`dpd`, `dpd_max_6m`, `dpd_max_12m`, `dpd_mean_12m`, `n_30p_6m`, `n_30p_12m`, `n_60p_12m`, `n_cure_12m`, `months_since_30p`, `ever_30p`) в кандидаты не входят: пропущенный платёж — это уже видимый шаг к дефолту, поэтому такие признаки слишком близки к самому событию (почти утечка). Модель должна предупреждать о *скрытом ухудшении* — до того, как появится какая-либо просрочка. Для каждого из 59 кандидатов *до того, как смотреть на связь с целевой переменной*, записываем: группу, экономический смысл и **ожидаемый знак** связи с риском. В следующих ноутбуках фактический знак сравнивается с этим ожиданием. Если они не совпали, мы разбираемся (утечка, взаимодействие, ошибка данных), но ожидание не переписываем.

**Sign convention / Обозначения знака:**
- `+` higher value → higher risk / чем больше значение, тем выше риск;
- `-` higher value → lower risk / чем больше значение, тем ниже риск;
- `cat` categorical: the `risky` level should have a higher event rate than the `safe` level / категориальный: у уровня `risky` доля событий должна быть выше, чем у `safe`;
- `none` no prior (exposure size, geography, institution names) / знак заранее не задаётся (размер кредита, география, названия организаций).

The table lives in `src/xai_ews/features/rationale.py`; this notebook only displays it. / Таблица хранится в `src/xai_ews/features/rationale.py`, ноутбук её только показывает.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import gc
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams.update({"figure.dpi": 80, "axes.grid": True, "grid.alpha": 0.3})

from xai_ews.features.rationale import rationale_table
from xai_ews.paths import FEATURE_SELECTION_DIR

FEATURE_SELECTION_DIR.mkdir(parents=True, exist_ok=True)
rat = rationale_table()
rat.to_csv(FEATURE_SELECTION_DIR / "feature_rationale.csv", index=False)
print(len(rat), "candidate features")

59 candidate features


### 1. Groups and signs at a glance / Группы и знаки — сводка

**EN.** How many candidates each group has and which signs we expect. Groups: *borrower* and *loan* — fixed at origination; *geography*; *exposure* — loan size (balance); *state_at_t* — the loan's state at observation date t (rate, age, remaining term, servicer); *behaviour* — how the loan was repaid over the last 6–12 months (principal paydown, partial prepayments, balance increases, modifications; delinquency history is not a candidate); *macro* — house prices, unemployment and market rates (with publication lags).

**RU.** Сколько кандидатов в каждой группе и какие знаки ожидаются. Группы: *borrower* (заёмщик) и *loan* (кредит) — данные на момент выдачи; *geography* — география; *exposure* — размер кредита (остаток долга); *state_at_t* — состояние кредита на дату наблюдения t (ставка, возраст, остаток срока, сервисер); *behaviour* — как кредит гасился за последние 6–12 месяцев (погашение основного долга, частичные досрочные погашения, рост остатка, модификации; история просрочек в кандидаты не входит); *macro* — цены на жильё, безработица и рыночные ставки (с учётом лагов публикации).

In [2]:
pd.crosstab(rat["group"], rat["expected_sign"], margins=True, margins_name="total")

expected_sign,+,-,cat,none,total
group,,,,,
behaviour,6,4,0,1,11
borrower,1,3,1,0,5
exposure,0,0,0,3,3
geography,0,0,0,3,3
loan,6,0,11,3,20
macro,8,4,0,0,12
state_at_t,4,0,0,1,5
total,25,11,12,11,59


**EN. How to read.** Of the 11 behavioural features, 6 have sign `+`: slower principal repayment (`upb_chg_6m`, `upb_chg_12m`, `upb_vs_schedule`), months with a balance increase (`n_upb_increase_12m`) and the modification flags (`modified_now`, `ever_modified`). Four have sign `-`: partial prepayments (`n_curtail_*`, `excess_principal_12m`) signal spare money, and `rate_change` is negative when the rate was cut in a modification. `n_dpd_unknown_12m` has no prior. In *state_at_t* the rate, loan age and remaining term have `+`, the servicer has no prior. Exposure, geography and institution names have no prior (`none`) — following methodology section 4, the loan size enters through the loss weight and EL, not through an assumed direction. In total: 25 features `+`, 11 `-`, 12 categorical, 11 without a prior.

**RU. Как читать.** Из 11 поведенческих признаков у 6 знак `+`: более медленное погашение основного долга (`upb_chg_6m`, `upb_chg_12m`, `upb_vs_schedule`), месяцы с ростом остатка (`n_upb_increase_12m`) и флаги модификации (`modified_now`, `ever_modified`). У четырёх знак `-`: частичные досрочные погашения (`n_curtail_*`, `excess_principal_12m`) говорят о свободных деньгах, а `rate_change` отрицателен, когда ставку снизили при модификации. Для `n_dpd_unknown_12m` знак не задаётся. В группе *state_at_t* у ставки, возраста кредита и остатка срока знак `+`, у сервисера знак не задаётся. Для экспозиции, географии и названий организаций знак не задаётся (`none`): по разделу 4 методологии размер кредита учитывается через вес в функции потерь и EL, а не через заранее заданное направление. Всего: 25 признаков со знаком `+`, 11 со знаком `-`, 12 категориальных, 11 без заданного знака.

### 2. The full table / Полная таблица

**EN.** All 59 candidates with meaning and expected sign. `static_h3 = True` marks the **static set** (origination data + geography, 28 features): for hypothesis H3 we compare a Y1 model on this set with a model on all selected features. The rationale columns in English (`why_en`) are in the CSV `reports/feature_selection/feature_rationale.csv`.

**RU.** Все 59 кандидатов со смыслом и ожидаемым знаком. `static_h3 = True` отмечает **статический набор** (данные при выдаче + география, 28 признаков): для гипотезы H3 модель Y1 на этом наборе сравнивается с моделью на всех отобранных признаках. Обоснование на английском (`why_en`) — в CSV `reports/feature_selection/feature_rationale.csv`.

In [3]:
show = rat[["feature", "group", "static_h3", "expected_sign", "risky_level", "safe_level", "meaning_ru", "why_ru"]].copy()
show["feature"] = show["feature"].str.slice(0, 45)
with pd.option_context("display.max_rows", 80, "display.max_colwidth", 70):
    display(show.set_index("feature"))

,group,static_h3,expected_sign,risky_level,safe_level,meaning_ru,why_ru
feature,,,,,,,
borrower_credit_score_at_origination,borrower,True,-,NaN,NaN,кредитный рейтинг (FICO) заёмщика при выдаче,отражает историю платежей; классический сильнейший предиктор
co_borrower_credit_score_at_origination,borrower,True,-,NaN,NaN,"рейтинг созаёмщика (пусто, если его нет)",второй надёжный плательщик снижает риск
debt_to_income_dti,borrower,True,+,NaN,NaN,"долговая нагрузка: платежи по долгам / доход, %",меньше свободного дохода — меньше запас прочности при шоке
number_of_borrowers,borrower,True,-,NaN,NaN,число заёмщиков по кредиту,два дохода снижают риск потери платёжеспособности
first_time_home_buyer_indicator,borrower,True,cat,Y,N,покупает жильё впервые (Y/N),меньше накоплений и нет опыта владения жильём
original_interest_rate,loan,True,+,NaN,NaN,"ставка по кредиту при выдаче, %",ставка включает надбавку за риск и увеличивает платёж
original_upb,exposure,False,none,NaN,NaN,сумма кредита при выдаче,"экспозиция: учитывается через вес ω и EL, знак заранее не задаётся..."
original_loan_term,loan,True,+,NaN,NaN,"срок кредита по договору, мес.",заёмщики на 15 лет обычно сильнее; 30-летний кредит медленнее гасится
original_loan_to_value_ratio_ltv,loan,True,+,NaN,NaN,"кредит / стоимость жилья при выдаче (LTV), %","меньше собственных средств, больше стимул к дефолту при падении цен"


### 3. Static set for H3 / Статический набор для H3

**EN.** Static features are known at origination and never change for the loan (borrower, loan terms, geography). They describe *who* borrowed; dynamic features (behaviour, state at t, exposure at t, macro) describe *what is happening now*. Among the 59 candidates 28 are static and 31 dynamic. H3 says that the dynamic features add a significant signal on top of the static ones for early default (Y1): deterioration of a loan that is still current shows in its repayment behaviour and its environment (house prices, unemployment) before any delinquency is visible.

**RU.** Статические признаки известны при выдаче и не меняются (заёмщик, условия кредита, география). Они описывают, *кто* взял кредит; динамические (поведение, состояние на t, экспозиция на t, макро) — *что происходит сейчас*. Из 59 кандидатов 28 статических и 31 динамический. H3 утверждает, что для раннего дефолта (Y1) динамические признаки добавляют значимый сигнал к статическим: ухудшение кредита, по которому ещё нет просрочки, видно в том, как он гасится, и в его окружении (цены на жильё, безработица) раньше, чем появится какая-либо просрочка.

In [4]:
static = rat[rat["static_h3"]]
print(f"static: {len(static)}, dynamic: {len(rat) - len(static)}")
with pd.option_context("display.max_colwidth", 400):
    display(static.groupby("group")["feature"].apply(", ".join).to_frame("static features"))

static: 28, dynamic: 31


,static features
group,
borrower,"borrower_credit_score_at_origination, co_borrower_credit_score_at_origination, debt_to_income_dti, number_of_borrowers, first_time_home_buyer_indicator"
geography,"property_state, metropolitan_statistical_area_msa_or_metropolitan_statistical_division_area_msda, zip_code_short"
loan,"original_interest_rate, original_loan_term, original_loan_to_value_ratio_ltv, original_combined_loan_to_value_ratio_cltv, loan_purpose, channel, seller_name, property_type, number_of_units, occupancy_status, mortgage_insurance_percentage, mortgage_insurance_type, amortization_type, prepayment_penalty_indicator, interest_only_loan_indicator, high_balance_loan_indicator, property_valuation_metho..."


### 4. Label-free profile on train / Профиль признаков на train без меток

**EN.** Before computing any link with the target we check, on train only, how many distinct values each feature has and how often it is missing. This needs no labels, so it does not bias the expected signs. Features with a single value cannot carry any information and will drop out in step 2 automatically. Numbers are weighted by 1/r, i.e. they describe the full train population.

**RU.** Перед расчётом связи с целевой переменной проверяем на train, сколько разных значений у признака и как часто он пропущен. Для этого метки не нужны, поэтому это не влияет на ожидаемые знаки. Признак с одним значением не несёт информации и автоматически отсеется на шаге 2. Цифры взвешены на 1/r, то есть относятся ко всей популяции train.

In [5]:
from xai_ews.data.features import candidate_features, categorical_features
from xai_ews.features.data import NA, load_train

cats = set(categorical_features())
tr = load_train("y1")            # Y1 population; labels are not used in this cell
w = tr["sample_weight"].to_numpy()
rows = []
for f in candidate_features():
    s = tr[f]
    miss = (s == NA) if f in cats else s.is_null() | s.is_nan()
    top_share = (tr.group_by(f).agg(pl.col("sample_weight").sum()).get_column("sample_weight").max() / w.sum())
    rows.append({"feature": f, "n_unique": s.n_unique(), "missing_share": float((miss.to_numpy() * w).sum() / w.sum()),
                 "top_value_share": float(top_share)})
profile = pd.DataFrame(rows)
del tr; gc.collect()
print("single-valued on train:", profile.query("n_unique == 1")["feature"].tolist())
print("top value covers > 99.5 %:", profile.query("n_unique > 1 and top_value_share > 0.995")["feature"].tolist())
profile.sort_values("missing_share", ascending=False).head(8).set_index("feature")

single-valued on train: ['amortization_type', 'property_valuation_method', 'special_eligibility_program', 'high_loan_to_value_hltv_refinance_option_indicator', 'n_dpd_unknown_12m']
top value covers > 99.5 %: ['prepayment_penalty_indicator', 'interest_only_loan_indicator', 'rate_change', 'modified_now', 'ever_modified']


,n_unique,missing_share,top_value_share
feature,,,
property_valuation_method,1,1.0000,1.0000
mortgage_insurance_type,3,0.8327,0.8327
mortgage_insurance_percentage,26,0.8327,0.8327
co_borrower_credit_score_at_origination,252,0.4282,0.4282
upb_chg_12m,1318527,0.2534,0.2534
upb_chg_6m,1684787,0.0107,0.0107
borrower_credit_score_at_origination,276,0.0005,0.0185
debt_to_income_dti,64,0.0002,0.0402


**EN. Conclusion.**
- Five fields have **one value** in train: all loans are fixed-rate (`amortization_type`), the HLTV refinance option and the special eligibility programs are not used, `property_valuation_method` is empty (the field is filled only in later vintages), and the delinquency status is never unknown (`n_dpd_unknown_12m` = 0). They cannot be selected.
- Five more are **almost constant** (the top value covers > 99.5 %): prepayment penalty and interest-only flags (a handful of missing values only) and the modification flags `modified_now`, `ever_modified` and `rate_change` (loans of the 2012–2014 vintages were almost never modified by 2015). They stay in the analysis: a rare state can still be very informative, IV decides. Modified loans are now very few: almost all of them had been in default before, and such loans are excluded from the population, so in 02_2 the modification flags form a single bin.
- *Missing has a meaning* for several features, and WoE binning gives it a separate bin: `co_borrower_credit_score_at_origination` is empty without a co-borrower (43 %), `mortgage_insurance_*` without insurance (83 %), `upb_chg_12m` if the balance 12 months earlier is not observed (young loans, 25 %).

**RU. Вывод.**
- У пяти полей в train **одно значение**: все кредиты с фиксированной ставкой (`amortization_type`), опция HLTV-рефинансирования и льготные программы не используются, `property_valuation_method` пуст (поле заполняется только в более поздних поколениях), статус просрочки ни разу не был неизвестен (`n_dpd_unknown_12m` = 0). Отобраны они быть не могут.
- Ещё пять **почти постоянны** (самое частое значение > 99,5 %): флаги штрафа за досрочное погашение и «только проценты» (лишь несколько пропусков) и флаги модификации `modified_now`, `ever_modified` и `rate_change` (кредиты поколений 2012–2014 к 2015 г. почти не реструктурировались). Они остаются в анализе: редкое состояние может быть очень информативным, решает IV. Модифицированных кредитов теперь очень мало: почти все они раньше были в дефолте, а такие кредиты исключены из популяции, поэтому в 02_2 у флагов модификации одна корзина.
- У нескольких признаков *пропуск имеет смысл*, и WoE-биннинг выносит его в отдельную корзину: `co_borrower_credit_score_at_origination` пуст без созаёмщика (43 %), `mortgage_insurance_*` — без страховки (83 %), `upb_chg_12m` — если ОД 12 месяцев назад не наблюдается (молодые кредиты, 25 %).